In [1]:
import fastf1
import pandas as pd 
import os


if not os.path.exists('../cache'):
    os.makedirs('../cache')

fastf1.Cache.enable_cache('../cache')

print("Laboratorio configurado e cache do FastF1 ativado com sucesso!")

Laboratorio configurado e cache do FastF1 ativado com sucesso!


In [2]:
def extrair_telemetria_piloto(ano,gp,piloto):
    session = fastf1.get_session(ano, gp, 'R') #Puxa os dados da sessao de corrida escolhida dentro do parenteses
    session.load(telemetry=False, weather=False)  # Carrega os dados da sessão sem telemetria e clima
    
    voltas_piloto = session.laps.pick_drivers(piloto).pick_accurate() #Filtra apenas as voltas do piloto escolhido dentro do parenteses

    dados_modeloML = voltas_piloto[['LapNumber', 'Stint', 'Compound', 'TyreLife', 'LapTime']].copy()
    dados_modeloML['LapTime'] = dados_modeloML['LapTime'].dt.total_seconds() 

    tempo_minimo = dados_modeloML['LapTime'].min() #Pega o tempo minimo de volta do piloto escolhido
    limite_tempo = tempo_minimo * 1.10
    dados_modeloML = dados_modeloML[dados_modeloML['LapTime'] <= limite_tempo] #Filtra apenas as voltas com tempo menor que 10% do tempo minimo, para evitar outliers

    compostos_esperados = ['Compound_SOFT', 'Compound_MEDIUM', 'Compound_HARD', 'Compound_INTERMEDIATE', 'Compound_WET']
    for composto in compostos_esperados:
        if composto not in dados_modeloML.columns:
            dados_modeloML[composto] = 0
            
    return dados_modeloML